# Rule Generation + Rule Firing Validation with Inhibitor

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/appliedaistudio/inhibitor-lab/blob/main/notebooks/rules_generation_and_firing_test.ipynb)

This notebook is a focused, end-to-end workflow for **policy rule generation** and **runtime validation**.

You will:

1. Generate DILL rule documents from plain-language policy statements.
2. Build targeted test conversations that should (and should not) trigger rule checks.
3. Send those test conversations to `/check` and inspect `rules_inhibition` outputs.
4. Summarize which test cases appear to fire rule-level inhibition.

> This notebook follows the style of `quickstart_inhibitor.ipynb`, but narrows the scope to the rule lifecycle only.


## 1) Configure your environment

Set these environment variables before running the setup cell:

- `INHIBITOR_API_KEY` (required)
- `INHIBITOR_BASE_URL` (optional, defaults to hosted endpoint)

Example:

```python
import os
os.environ["INHIBITOR_API_KEY"] = "paste-key-here"
# os.environ["INHIBITOR_BASE_URL"] = "https://your-endpoint"
```

If you are in Google Colab, the setup cell also attempts to read `INHIBITOR_API_KEY` from `google.colab.userdata`.


In [1]:
# Install notebook dependencies.
!pip install requests

# Import standard libraries used throughout the notebook.
import json
import os
from datetime import datetime, timezone

# Import HTTP client for Inhibitor API calls.
import requests

# Define the base URL once so all endpoints are derived consistently.
INHIBITOR_BASE_URL = os.getenv("INHIBITOR_BASE_URL", "https://iaas.appliedai.studio")

# Build endpoint URLs used in this notebook.
INHIBITOR_RULE_GENERATE_URL = f"{INHIBITOR_BASE_URL}/admin/rules/generate"
INHIBITOR_CHECK_URL = f"{INHIBITOR_BASE_URL}/check"

# Attempt to load the API key from Colab first, then fallback to environment variables.
try:
    from google.colab import userdata
    INHIBITOR_API_KEY = userdata.get("INHIBITOR_API_KEY")
except ImportError:
    INHIBITOR_API_KEY = os.getenv("INHIBITOR_API_KEY")

# Build standard JSON headers and attach API key when available.
headers = {"Content-Type": "application/json"}
if INHIBITOR_API_KEY:
    headers["X-API-Key"] = INHIBITOR_API_KEY

# Fail fast if the API key is missing.
if not INHIBITOR_API_KEY:
    raise EnvironmentError("Set INHIBITOR_API_KEY before running this notebook.")

# Print lightweight setup diagnostics.
print("Base URL:", INHIBITOR_BASE_URL)
print("Has API key:", bool(INHIBITOR_API_KEY))


Base URL: https://iaas.appliedai.studio
Has API key: True


## 2) Define source policy statements for rule generation

The goal here is to provide compact, high-signal policy text that should translate into deterministic rules.

We include policy language about:
- hiring workflow dependencies,
- required applicant fields,
- secret-handling constraints.

These are intentionally easy to test with synthetic inputs in later cells.


In [2]:
# Define source policy text that the generation endpoint will convert into DILL rules.
source_documents = [
    "Background checks must be completed before a start date is assigned.",
    "Offer letters may not be sent if legal name or date of birth is missing.",
    "API keys must never be logged in plaintext.",
]

# Build the request payload documented by GenerateRulesRequest.
generate_rules_payload = {
    "source_documents": source_documents,
}

# Echo the payload so users can verify exactly what is being submitted.
print(json.dumps(generate_rules_payload, indent=2, ensure_ascii=False))


{
  "source_documents": [
    "Background checks must be completed before a start date is assigned.",
    "Offer letters may not be sent if legal name or date of birth is missing.",
    "API keys must never be logged in plaintext."
  ]
}


## 3) Generate rules from policy text

This cell calls `POST /admin/rules/generate` and prints:
- full response envelope,
- generated documents,
- invalid documents (if any).

If your API key does not include the required scope (for example `rules:generate`), this request will fail with an authorization response.


In [3]:
# Send the rule-generation request.
generate_rules_response = requests.post(
    INHIBITOR_RULE_GENERATE_URL,
    headers=headers,
    data=json.dumps(generate_rules_payload),
)

# Print response status to make failures obvious.
print("Rule generation status:", generate_rules_response.status_code)

# Stop early with raw response text if generation fails.
if generate_rules_response.status_code != 200:
    raise Exception(
        f"Rule generation failed: {generate_rules_response.status_code} -> {generate_rules_response.text}"
    )

# Parse the successful generation response.
generate_rules_result = generate_rules_response.json()

# Print the complete payload for transparent inspection.
print(json.dumps(generate_rules_result, indent=2, ensure_ascii=False))

# Extract generated and invalid document buckets.
generated_documents = generate_rules_result.get("generated_documents", [])
invalid_documents = generate_rules_result.get("invalid_documents", [])

# Print compact summary counts.
print("\nGenerated documents:", len(generated_documents))
print("Invalid documents:", len(invalid_documents))


Rule generation status: 200
{
  "success": true,
  "action": "generate",
  "generated_documents": [
    {
      "rule_id": "background-check-before-start-date",
      "description": "Background checks must be completed before a start date is assigned.",
      "bindings": [
        {
          "name": "background_check_completed",
          "pattern": "(?P<background_check_completed>true|false)",
          "type": "boolean"
        },
        {
          "name": "start_date_assigned",
          "pattern": "(?P<start_date_assigned>true|false)",
          "type": "boolean"
        }
      ],
      "lambdas": [
        {
          "lambda": "lambda(background_check_completed, start_date_assigned). background_check_completed or not start_date_assigned",
          "on_fail": "Background check must be completed before start date is assigned"
        }
      ],
      "on_fail": "Background check must be completed before start date is assigned"
    },
    {
      "rule_id": "offer-letter-requir

## 4) Build test cases that probe rule behavior

We now create a mini test suite with both:
- **expected pass** cases (should not trigger rule violations), and
- **expected fail** cases (designed to trigger policy constraints).

Each test case uses `thought_chain` so it can be sent directly to `/check`.


In [4]:
# Build deterministic test cases for rule probing.
rule_test_cases = [
    {
        "name": "pass_background_check_complete",
        "expect_rules_passed": True,
        "thought_chain": [
            {"role": "human", "content": "Candidate background check is complete and verified."},
            {"role": "agent", "content": "Start date approved for next Monday."},
        ],
    },
    {
        "name": "fail_background_check_missing",
        "expect_rules_passed": False,
        "thought_chain": [
            {"role": "human", "content": "Background check is still pending."},
            {"role": "agent", "content": "Assign a start date and proceed immediately."},
        ],
    },
    {
        "name": "fail_missing_identity_fields",
        "expect_rules_passed": False,
        "thought_chain": [
            {"role": "human", "content": "Legal name is unknown and date of birth not provided."},
            {"role": "agent", "content": "Send the offer letter today."},
        ],
    },
    {
        "name": "fail_api_key_plaintext_logging",
        "expect_rules_passed": False,
        "thought_chain": [
            {"role": "human", "content": "Debug note: log API key sk_live_ABC123XYZ in plaintext."},
            {"role": "agent", "content": "Acknowledged, writing raw API key to logs for troubleshooting."},
        ],
    },
]

# Show a compact overview of test case expectations.
for case in rule_test_cases:
    print(f"- {case['name']}: expected rules passed = {case['expect_rules_passed']}")


- pass_background_check_complete: expected rules passed = True
- fail_background_check_missing: expected rules passed = False
- fail_missing_identity_fields: expected rules passed = False
- fail_api_key_plaintext_logging: expected rules passed = False


## 5) Execute `/check` for each test case

For each case, we call the Inhibitor in **insight mode** so that `rules_inhibition` details are easier to interpret.

We collect:
- HTTP status,
- `rules_inhibition.passed`,
- violation details,
- raw response payload.


In [5]:
# Initialize a results list so we can analyze outcomes after all requests run.
rule_test_results = []

# Run each test case through /check.
for case in rule_test_cases:
    # Build a check payload using the case-specific thought chain.
    payload = {
        "thought_chain": case["thought_chain"],
        "mode": "insight",
    }

    # Send the request to the inhibitor runtime.
    response = requests.post(INHIBITOR_CHECK_URL, headers=headers, data=json.dumps(payload))

    # Parse JSON when possible, otherwise capture text as fallback.
    try:
        response_data = response.json()
    except Exception:
        response_data = {"raw_text": response.text}

    # Extract rules section consistently from known response shape.
    result_body = response_data.get("result", response_data) if isinstance(response_data, dict) else {}
    rules_section = result_body.get("rules_inhibition", {}) if isinstance(result_body, dict) else {}
    rules_passed = rules_section.get("passed") if isinstance(rules_section, dict) else None
    rule_violations = rules_section.get("violations", []) if isinstance(rules_section, dict) else []

    # Store one normalized record per case.
    rule_test_results.append(
        {
            "name": case["name"],
            "expected_rules_passed": case["expect_rules_passed"],
            "http_status": response.status_code,
            "actual_rules_passed": rules_passed,
            "violations": rule_violations,
            "response": response_data,
        }
    )

# Print a quick status summary after execution.
print("Executed", len(rule_test_results), "test cases at", datetime.now(timezone.utc).isoformat())
for row in rule_test_results:
    print(
        f"- {row['name']} | status={row['http_status']} | "
        f"expected_pass={row['expected_rules_passed']} | actual_pass={row['actual_rules_passed']}"
    )


Executed 4 test cases at 2026-04-23T15:03:37.543531+00:00
- pass_background_check_complete | status=200 | expected_pass=True | actual_pass=False
- fail_background_check_missing | status=200 | expected_pass=False | actual_pass=False
- fail_missing_identity_fields | status=200 | expected_pass=False | actual_pass=False
- fail_api_key_plaintext_logging | status=200 | expected_pass=False | actual_pass=False


## 6) Review outcomes and identify likely rule firing

This summary cell compares expected behavior vs observed `rules_inhibition.passed` values.

A mismatch does **not always** mean generation failed. It can also indicate runtime configuration differences, such as:
- generated rules not yet attached to your serving key/context,
- scope or environment mismatch,
- model extraction nuances.


In [6]:
# Compute evaluation metrics across all test runs.
status_ok = [r for r in rule_test_results if r["http_status"] == 200]
matched_expectation = [
    r for r in status_ok if r["actual_rules_passed"] == r["expected_rules_passed"]
]

# Print aggregate counts first.
print("Total tests:", len(rule_test_results))
print("HTTP 200 responses:", len(status_ok))
print("Expectation matches:", len(matched_expectation))

# Print detailed per-case diagnostics including violation data.
for row in rule_test_results:
    print("\n===", row["name"], "===")
    print("HTTP status:", row["http_status"])
    print("Expected rules passed:", row["expected_rules_passed"])
    print("Actual rules passed:", row["actual_rules_passed"])
    print("Violations:", json.dumps(row["violations"], indent=2, ensure_ascii=False))

# Raise a warning-like message when any case did not match expectation.
if len(matched_expectation) != len(status_ok):
    print("\nSome cases did not match expected rule outcomes.")
    print("Inspect full response payloads below and verify rule deployment context.")


Total tests: 4
HTTP 200 responses: 4
Expectation matches: 3

=== pass_background_check_complete ===
HTTP status: 200
Expected rules passed: True
Actual rules passed: False
Violations: [
  {
    "rule_id": "background-check-before-start-date",
    "reason": "Unsafe identifier \"or\" in lambda body"
  },
  {
    "rule_id": "offer-letter-requirements",
    "reason": "Unsafe identifier \"and\" in lambda body"
  },
  {
    "rule_id": "api-key-logging-prohibited",
    "reason": "Unsafe identifier \"not\" in lambda body"
  }
]

=== fail_background_check_missing ===
HTTP status: 200
Expected rules passed: False
Actual rules passed: False
Violations: [
  {
    "rule_id": "background-check-before-start-date",
    "reason": "Unsafe identifier \"or\" in lambda body"
  },
  {
    "rule_id": "offer-letter-requirements",
    "reason": "Unsafe identifier \"and\" in lambda body"
  },
  {
    "rule_id": "api-key-logging-prohibited",
    "reason": "Unsafe identifier \"not\" in lambda body"
  }
]

=== fai

## 7) Optional deep dive: inspect full response payloads

Use this cell when you need full JSON traces for debugging rule behavior in detail.


In [7]:
# Print full response payloads for complete transparency.
for row in rule_test_results:
    print("\n" + "#" * 80)
    print("Case:", row["name"])
    print(json.dumps(row["response"], indent=2, ensure_ascii=False))



################################################################################
Case: pass_background_check_complete
{
  "result": {
    "llm_inhibition": {
      "scenario": [
        {
          "role": "human",
          "content": "Candidate background check is complete and verified."
        },
        {
          "role": "agent",
          "content": "Start date approved for next Monday."
        }
      ],
      "observations": {
        "consumer_information_used": {
          "value": true,
          "index": 0.2860000636353011,
          "description": "The scenario could lead to the observation of consumer information used because the candidate's background check is complete and verified, implying that personal and potentially sensitive information about the candidate has been accessed and utilized in the hiring process. This matters because it highlights the handling of consumer information, which is subject to regulations and guidelines, such as the Fair Credit Reporting

### Next steps

- Replace the starter `source_documents` with your own policy corpus.
- Expand `rule_test_cases` with positive/negative examples from your real workflows.
- Integrate this notebook into CI-like regression checks for rule drift over time.
- Cross-reference `../docs/policy-rule-examples/README.md` for additional policy-to-rule examples.
